# C/G-rich parent RNA screen: at least 8 C/G in positions −15 to −2

This notebook implements the revised C/G-rich screen using polyA_DB v3.2 and UCSC hg19:

- Select annotated human 3′UTR sites marked `Conservation = Yes`.
- Use the final 15 bases of the extracted 50-nt UTR window followed by A15. The −1 nucleotide, immediately before A15, must be U, C or G.
- At least **8 of the 14 nucleotides from −15 through −2** must be C or G (combined). The −1 nucleotide is excluded from this count. At least **25 of the 50-nt UTR window** must be C or G.
- None of the five target A/AU motifs (`AAAAA`, `AAAAU`, `UAAAA`, `UAUAA`, `AUAUA`) may occur wholly inside −15 through −2.
- Screen the target C/G motifs (`GGCCG`, `CCGCG`, `GCGCG`, `CGGCG`, `CGCGG`) at −13 to −9, −11 to −7, and −6 to −2. Select up to two per motif and position; full coverage of all five motifs is not required.
- Check a separate both-ends category requiring target motifs at both −15 to −11 and −6 to −2. The two motifs may be the same. Rank by the A count in the first 35 bases of the 50-nt UTR, descending; PAS ID breaks ties.

The synthetic A30 control is listed separately because it has no non-A position −1. Non-conserved sites appear only as excluded diagnostics, never as selected parents. Empty categories mean no eligible sequences were found; the code does not relax conditions automatically.

This reconstructs a screen from polyA_DB and hg19. It is not Müller’s unpublished 38,700-sequence list; the genomic windows are contiguous and not splice-aware. A 15-nt modeled segment does not establish the full biological 3′UTR length. This notebook does not calculate Gibbs free energy.

Run the cells in order. The first run downloads the database and the approximately 816 MB hg19 reference, verifies the files, and downloads a ZIP with the Excel workbook.


## 1. Install packages

In [ ]:
import sys, subprocess
try:
    import py2bit
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py2bit==1.0.1'], check=True)
    import py2bit
try:
    import openpyxl
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'], check=True)
    import openpyxl
print('Packages ready.')


## 2. Define the database extraction rules

In [ ]:
import csv, hashlib
from decimal import Decimal, InvalidOperation
from collections import Counter
from itertools import product
from pathlib import Path
from datetime import datetime

CG_MOTIFS = ('GGCCG', 'CCGCG', 'GCGCG', 'CGGCG', 'CGCGG')

AU_MOTIFS = ('AAAAA', 'AAAAU', 'UAAAA', 'UAUAA', 'AUAUA')

CONTROL_ID = 'chr5:134907433:-'

CONTROL_SEQUENCE = 'A' * 30

PAS_SHA256 = 'c20167a841763b2977376467ef80014765ef595db93f2a023fb2d05a9c39b73d'

HG19_MD5 = 'bcdbfbe9da62f19bee88b74dabef8cd3'

HG19_BYTES = 816241703

COMPLEMENT = str.maketrans('ACGT', 'TGCA')

def digest(path, name):
    h = hashlib.new(name)
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def extract50(genome, chrom, pos, strand):
    chroms = genome.chroms()
    if chrom not in chroms:
        raise ValueError('chromosome_missing')
    if strand == '+':
        start, end = pos - 50, pos
    elif strand == '-':
        start, end = pos - 1, pos + 49
    else:
        raise ValueError('invalid_strand')
    if start < 0 or end > chroms[chrom]:
        raise ValueError('chromosome_boundary')
    seq = genome.sequence(chrom, start, end)
    if seq is None or len(seq) != 50:
        raise ValueError('reference_read_failed_or_short')
    seq = seq.upper()
    if set(seq) - set('ACGT'):
        raise ValueError('ambiguous_reference_bases')
    if strand == '-':
        seq = seq.translate(COMPLEMENT)[::-1]
    return seq.replace('T', 'U')

def select_panel(rows, motifs, target, excluded_sequence=None):
    """Four per group where available; fill shortages by global score.

    A single 30-nt sequence contributes at most once; the highest scoring
    source PAS is retained, then lexical PAS ID breaks ties.
    """
    candidates = sorted((dict(r) for r in rows if r['motif'] in motifs
                         and r['utr15_A15'] != excluded_sequence),
                        key=lambda r: (-r['score'], r['PAS_ID']))
    unique, seen = [], set()
    for r in candidates:
        if r['utr15_A15'] not in seen:
            seen.add(r['utr15_A15']); unique.append(r)
    cap = target // len(motifs)
    result, selected = [], set()
    for motif in motifs:
        group = [r for r in unique if r['motif'] == motif]
        for r in group[:cap]:
            result.append(r); selected.add(r['utr15_A15'])
    for r in unique:
        if len(result) >= target: break
        if r['utr15_A15'] not in selected:
            result.append(r); selected.add(r['utr15_A15'])
    return result[:target]

def read_sites(path):
    """Collect each annotated 3'UTR PAS once; retain conservative membership."""
    groups = {}
    n_rows = 0
    with open(path, newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f, delimiter='\t')
        required = {'PAS_ID','Chromosome','Position','Strand','Mean RPM',
                    'Conservation','PAS type','Gene Symbol'}
        missing = required - set(reader.fieldnames or [])
        if missing: raise ValueError(f'Missing columns in PAS table: {sorted(missing)}')
        for r in reader:
            n_rows += 1
            if not r['PAS type'].strip().startswith("3'UTR("):
                continue
            pas, chrom, strand = r['PAS_ID'].strip(), r['Chromosome'].strip(), r['Strand'].strip()
            try:
                pos, rpm = int(r['Position']), Decimal(r['Mean RPM'])
            except (ValueError, InvalidOperation) as e:
                raise ValueError(f'Invalid coordinate or mean RPM: {pas}') from e
            if pas != f'{chrom}:{pos}:{strand}' or strand not in ('+','-') or pos < 1 or not rpm.is_finite() or rpm < 0:
                raise ValueError(f'Invalid PAS annotation: {pas}')
            key = (chrom,pos,strand,rpm)
            g = groups.setdefault(pas, {'key':key, 'genes':set(), 'types':set(), 'conserved':False})
            if g['key'] != key: raise ValueError(f'Conflicting annotations for {pas}')
            symbol = r['Gene Symbol'].strip()
            if symbol and symbol.lower() not in ('nan','na','none'): g['genes'].add(symbol)
            g['types'].add(r['PAS type'].strip())
            g['conserved'] |= r['Conservation'].strip().lower() == 'yes'
    sites=[]
    for pas,g in groups.items():
        chrom,pos,strand,rpm=g['key']
        sites.append({'PAS_ID':pas,'Chromosome':chrom,'Position':pos,'Strand':strand,
                      'Mean_RPM':float(rpm),'Genes':';'.join(sorted(g['genes'])),
                      'PAS_types':';'.join(sorted(g['types'])),
                      'Conservation': 'Yes' if g['conserved'] else 'No'})
    return sites,n_rows

def unique_candidates(rows, excluded_sequence=None):
    seen=set();out=[]
    for r in sorted(rows,key=lambda x:(-x['score'],x['PAS_ID'])):
        if r['utr15_A15']==excluded_sequence or r['utr15_A15'] in seen: continue
        seen.add(r['utr15_A15']);out.append(r)
    return out


## 3. Download and verify inputs

In [ ]:
import os, shutil, urllib.request, zipfile

pas_path = Path('human.PAS.txt')
genome_path = Path('hg19.2bit')

def download_to(url, path):
    with urllib.request.urlopen(url, timeout=120) as response, open(path, 'wb') as dest:
        shutil.copyfileobj(response, dest, length=1024*1024)

if not pas_path.exists():
    print('Downloading the official polyA_DB v3.2 PAS table...')
    archive = Path('human_pas.zip.part')
    partial = Path('human.PAS.txt.part')
    try:
        download_to('https://exon.apps.wistar.org/PolyA_DB/v3/download/3.2/human_pas.zip', archive)
        with zipfile.ZipFile(archive) as z, z.open('human.PAS.txt') as source, partial.open('wb') as dest:
            shutil.copyfileobj(source, dest)
        if digest(partial, 'sha256') != PAS_SHA256:
            raise ValueError('Downloaded PAS table checksum does not match the expected v3.2 file.')
        os.replace(partial, pas_path)
    finally:
        archive.unlink(missing_ok=True)
        partial.unlink(missing_ok=True)
if digest(pas_path, 'sha256') != PAS_SHA256:
    raise ValueError('Existing human.PAS.txt has the wrong checksum. Rename or remove it and rerun.')

if not genome_path.exists():
    print('Downloading UCSC hg19.2bit (~816 MB). This may take several minutes...')
    partial = Path('hg19.2bit.part')
    try:
        download_to('https://hgdownload.soe.ucsc.edu/goldenPath/hg19/bigZips/hg19.2bit', partial)
        if partial.stat().st_size != HG19_BYTES or digest(partial, 'md5') != HG19_MD5:
            raise ValueError('Downloaded hg19 file failed checksum verification.')
        os.replace(partial, genome_path)
    finally:
        partial.unlink(missing_ok=True)
if genome_path.stat().st_size != HG19_BYTES or digest(genome_path, 'md5') != HG19_MD5:
    raise ValueError('Existing hg19.2bit failed verification. Rename or remove it and rerun.')
print('Both input files verified.')


## 4. Define the C/G screen and workbook

In [ ]:
"""C/G-rich screen of the normalized 15-nt parent RNA."""
import hashlib
from collections import defaultdict

CG_MOTIFS=('GGCCG','CCGCG','GCGCG','CGGCG','CGCGG')
AU_MOTIFS=('AAAAA','AAAAU','UAAAA','UAUAA','AUAUA')
SAMPLE_STARTS=(-13,-11,-6)
CONTROL_SEQUENCE='A'*30
FIELDS=['RNA_ID','Panel','selection_order','motif','norm_start','norm_end',
        'Genes','PAS_ID','Conservation','UTR50_A15','UTR50','utr15_A15','UTR15',
        'terminal_UTR_As','normalized_minus1_base','normalized_window_minus15_minus2',
        'upstream_A_count','upstream_A_pct','CG_count_50','CG_count_14','CG_count_15',
        'Mean_RPM','Chromosome','Position','Strand','PAS_types']

def valid_parent(seq,conserved):
 return (conserved and len(seq)==50 and set(seq)<=set('ACGU') and seq[-1] in 'CGU'
         and sum(c in 'CG' for c in seq[-15:-1])>=8
         and sum(c in 'CG' for c in seq)>=25
         and not any(m in seq[-15:-1] for m in AU_MOTIFS))

def motif_at(seq,start):
 if len(seq)!=50 or not -15<=start<=-6:raise ValueError('Invalid sequence or motif start')
 i=50+start
 return seq[i:i+5]

def both_ends(seq):
 far,near=motif_at(seq,-15),motif_at(seq,-6)
 return (far,near) if far in CG_MOTIFS and near in CG_MOTIFS else None

def choose_by_category(hits):
 out=[]
 for start in SAMPLE_STARTS:
  for motif in CG_MOTIFS:
   seen=set();group=[]
   for r in sorted((r for r in hits if r['norm_start']==start and r['motif']==motif),key=lambda r:(-r['score'],r['PAS_ID'])):
    seq=r['utr15_A15']
    if seq not in seen:seen.add(seq);group.append(r)
   out.extend({**r,'Panel':'Position top two','selection_order':i} for i,r in enumerate(group[:2],1))
 return out

def annotate(site,seq):
 tail=seq[-15:]+'A'*15;count=seq[:35].count('A')
 return {**site,'RNA_ID':'CG_RNA_N'+hashlib.sha256(tail.encode('ascii')).hexdigest()[:12],
         'UTR50':seq,'UTR50_A15':seq+'A'*15,'UTR15':seq[-15:],'utr15_A15':tail,
         'terminal_UTR_As':0,'normalized_minus1_base':seq[-1],
         'normalized_window_minus15_minus2':seq[-15:-1],
         'upstream_A_count':count,'upstream_A_pct':round(100*count/35,2),
         'CG_count_50':sum(seq.count(c) for c in 'CG'),
         'CG_count_14':sum(c in 'CG' for c in seq[-15:-1]),
         'CG_count_15':sum(seq[-15:].count(c) for c in 'CG'),'score':count}

def dedup(rows):
 seen=set();result=[]
 for r in sorted(rows,key=lambda r:(-r['score'],r['PAS_ID'])):
  key=r['utr15_A15']
  if key not in seen:seen.add(key);result.append(r)
 return result

def scan(sites,genome,extract50):
 hits=[];both=[];nonconserved=[];failures=[];rule_conserved_sites=0
 for site in sites:
  try:seq=extract50(genome,site['Chromosome'],site['Position'],site['Strand'])
  except ValueError as e:failures.append({'PAS_ID':site['PAS_ID'],'reason':str(e)});continue
  conserved=site['Conservation']=='Yes'
  if not valid_parent(seq,True):continue
  row=annotate(site,seq)
  if not conserved:
   nonconserved.append({**row,'exclusion_reason':'Conservation is No; excluded from selection'})
   continue
  rule_conserved_sites+=1
  for start in SAMPLE_STARTS:
   motif=motif_at(seq,start)
   if motif in CG_MOTIFS:hits.append({**row,'motif':motif,'norm_start':start,'norm_end':start+4})
  pair=both_ends(seq)
  if pair:both.append({**row,'Panel':'Both ends candidate','far_motif':pair[0],'near_motif':pair[1]})
 return {'hits':hits,'both':dedup(both),'nonconserved':nonconserved,'failures':failures,
         'rule_conserved_sites':rule_conserved_sites}

def write_workbook(sites,data,out,input_rows):
 from openpyxl import Workbook
 from openpyxl.styles import Font,PatternFill,Alignment
 from openpyxl.utils import get_column_letter
 selected=choose_by_category(data['hits']);parents=dedup(selected)
 for r in parents:
  cats=[x for x in selected if x['utr15_A15']==r['utr15_A15']]
  r['selected_categories']='; '.join(sorted({f"{x['norm_start']} to {x['norm_end']}: {x['motif']}" for x in cats}))
  r['both_ends_candidate']='Yes' if any(x['utr15_A15']==r['utr15_A15'] for x in data['both']) else 'No'
 counts=[]
 for start in SAMPLE_STARTS:
  for motif in CG_MOTIFS:
   hits=[x for x in data['hits'] if x['norm_start']==start and x['motif']==motif]
   picked=[x for x in selected if x['norm_start']==start and x['motif']==motif]
   counts.append({'norm_start':start,'norm_end':start+4,'motif':motif,'eligible_unique_RNAs':len(dedup(hits)),'selected_unique_RNAs':len(picked)})
 wb=Workbook();summary=wb.active;summary.title='Summary';summary.append(['Item','Value'])
 notes=[('Purpose','C/G-rich counterpart of the A/AU parent screen'),('Database rows',input_rows),('Unique annotated 3UTR sites',len(sites)),
        ('Sequence-rule conserved PASs',data['rule_conserved_sites']),('Distinct selected parents',len(parents)),('Selected motif-position rows',len(selected)),
        ('Distinct both-ends candidates',len(data['both'])),('Sequence-rule nonconserved PASs (excluded)',len(data['nonconserved'])),
        ('UTR and tail','Final 15 genomic bases plus appended A15; this does not imply the entire transcript UTR is 15 nt.'),
        ('Junction rule','Final genomic UTR base is C, G or U: position -1 is immediately adjacent to A15.'),
        ('Region rule','At least 8 of the 14 bases at -15 through -2 are C or G; position -1 is excluded from this count.'),
        ('50-nt rule','At least 25 of the 50 UTR bases are C or G.'),
        ('AU motif exclusion','None of AAAAA, AAAAU, UAAAA, UAUAA, AUAUA occurs wholly in -15 through -2.'),
        ('Conservation','Only PASs flagged conserved are eligible; non-conserved sites are diagnostic only.'),
        ('Targets','GGCCG, CCGCG, GCGCG, CGGCG, CGCGG; positions -13..-9, -11..-7, -6..-2.'),
        ('Selection','Up to two per motif and position; complete five-motif coverage not required.'),
        ('Both-ends rule','One target motif at -15..-11 and one at -6..-2; the motifs may be the same.'),
        ('Ranking','A count in the first 35 nt of 50-nt UTR, descending; ties by PAS ID. This matches the original A/AU parent ranking.'),
        ('Control','Synthetic 15 A + A15 = A30; no non-A anchor; listed separately, not a C/G parent.'),
        ('Provenance','polyA_DB v3.2; UCSC hg19 genomic windows; conserved 3UTR annotations. Not the unpublished Muller library; windows are contiguous and not splice-aware.'),
        ('Interpretation','Sequence screen only; no predicted or measured Gibbs binding free energy.')]
 for r in notes:summary.append(r)
 def add(name,rows,fields):
  ws=wb.create_sheet(name);ws.append(fields)
  for r in rows:ws.append([r.get(f,'') for f in fields])
  return ws
 add('Unique parent RNAs',parents,FIELDS+['selected_categories','both_ends_candidate'])
 add('Position top two',selected,FIELDS)
 add('All motif position candidates',data['hits'],FIELDS)
 add('Position counts',counts,['norm_start','norm_end','motif','eligible_unique_RNAs','selected_unique_RNAs'])
 add('Both ends candidates',data['both'],FIELDS+['far_motif','near_motif'])
 add('Both ends top two',[{**r,'selection_order':i,'Panel':'Both ends top two'} for i,r in enumerate(data['both'][:2],1)],FIELDS+['far_motif','near_motif'])
 add('Nonconserved excluded',data['nonconserved'],FIELDS+['exclusion_reason'])
 add('Extraction failures',data['failures'],['PAS_ID','reason'])
 add('All-A control',[{'RNA_ID':'CONTROL_A30','Panel':'Synthetic all-A control','UTR15':'A'*15,'utr15_A15':CONTROL_SEQUENCE,'note':'No non-A anchor; Muller -1 is undefined.'}],['RNA_ID','Panel','UTR15','utr15_A15','note'])
 for ws in wb:
  ws.freeze_panes='C2';ws.row_dimensions[1].height=30
  for c in ws[1]:c.font=Font(bold=True,color='FFFFFF');c.fill=PatternFill('solid',fgColor='173650');c.alignment=Alignment(wrap_text=True)
  if ws.max_row>1:ws.auto_filter.ref=ws.dimensions
  for i in range(1,ws.max_column+1):ws.column_dimensions[get_column_letter(i)].width=45 if ws.cell(1,i).value in ('UTR50','UTR50_A15','selected_categories','note') else 25
 wb.save(out)
 return {'selected':selected,'parents':parents,'counts':counts}


## 5. Run the screen and download the output

In [ ]:
import zipfile
from google.colab import files

print("Reading polyA_DB annotations...")
sites, input_rows = read_sites(pas_path)
print("Annotated 3′UTR sites:", len(sites))
reference = py2bit.open(str(genome_path))
try:
    result_data = scan(sites, reference, extract50)
finally:
    reference.close()
output_dir = Path("Muller_CG_8of14_results_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
output_dir.mkdir()
workbook = output_dir / "CG_8of14_parent_pool.xlsx"
selection = write_workbook(sites, result_data, workbook, input_rows)
print("Conserved sites meeting C/G-rich sequence rules:", result_data["rule_conserved_sites"])
print("Motif-position hit rows:", len(result_data["hits"]))
print("Distinct selected parents:", len(selection["parents"]))
print("Distinct both-ends candidates:", len(result_data["both"]))
print("Non-conserved sites excluded:", len(result_data["nonconserved"]))
print("Genome extraction failures:", len(result_data["failures"]))
if not selection["parents"]:
    print("NO ELIGIBLE PARENTS. The workbook records this result; no conditions have been relaxed.")
zip_path = output_dir / "CG_8of14_parent_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(workbook, arcname=workbook.name)
print("Downloading:", zip_path)
files.download(str(zip_path))
